# Code Assignment in Python - Optimized Version
This is a cleaned and optimized version of the original script.
Key improvements:
- Countries and tasks defined as lists — easy to add/remove
- Excel sheets loaded in a loop instead of 9 separate lines
- Standardization logic extracted into a reusable function
- Plots generated in a single loop

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Sets the path to the working directory
os.chdir("Z:\\File folders\\Teaching\\Reproducible Research\\2023\\Repository\\RRcourse2023\\6. Coding and documentation")

In [ ]:
# --- CONFIGURATION ---
# Add or remove countries and tasks here — the rest of the code adapts automatically

COUNTRIES = ["Belgium", "Spain", "Poland"]

# Non-routine cognitive analytical (NRCA) tasks as defined by Autor & Acemoglu
TASKS = ["t_4A2a4", "t_4A2b2", "t_4A4a1"]
# t_4A2a4 = Analyzing Data or Information
# t_4A2b2 = Thinking Creatively
# t_4A4a1 = Interpreting the Meaning of Information for Others

In [ ]:
# --- LOAD DATA ---

# Import O*NET task data (already crosswalked to ISCO-08)
task_data = pd.read_csv("Data\\onet_tasks.csv")

# Load all 9 ISCO sheets in a loop instead of 9 separate lines
isco_sheets = []
for i in range(1, 10):
    df = pd.read_excel("Data\\Eurostat_employment_isco.xlsx", sheet_name=f"ISCO{i}")
    df["ISCO"] = i
    isco_sheets.append(df)

all_data = pd.concat(isco_sheets, ignore_index=True)

In [ ]:
# --- CALCULATE EMPLOYMENT TOTALS AND SHARES ---

for country in COUNTRIES:
    # Total workers per time period across all ISCO categories
    total = sum(sheet[country] for sheet in isco_sheets)
    all_data[f"total_{country}"] = pd.concat([total] * 9, ignore_index=True)
    # Share of each occupation among all workers
    all_data[f"share_{country}"] = all_data[country] / all_data[f"total_{country}"]

In [ ]:
# --- PREPARE TASK DATA ---

# Keep only the first digit of the ISCO code
task_data["isco08_1dig"] = task_data["isco08"].astype(str).str[:1].astype(int)

# Aggregate task values to 1-digit ISCO level
aggdata = task_data.groupby("isco08_1dig").mean().drop(columns=["isco08"])

# Merge employment and task data
combined = pd.merge(all_data, aggdata, left_on="ISCO", right_on="isco08_1dig", how="left")

In [ ]:
# --- HELPER FUNCTION: WEIGHTED STANDARDIZATION ---

def weighted_standardize(series, weights):
    """Standardize a series using weighted mean and weighted std deviation."""
    mean = np.average(series, weights=weights)
    sd = np.sqrt(np.average((series - mean) ** 2, weights=weights))
    return (series - mean) / sd

In [ ]:
# --- STANDARDIZE TASKS AND COMPUTE NRCA ---

for country in COUNTRIES:
    weights = combined[f"share_{country}"]

    # Standardize each task
    for task in TASKS:
        combined[f"std_{country}_{task}"] = weighted_standardize(combined[task], weights)

    # NRCA = sum of standardized task values
    combined[f"{country}_NRCA"] = sum(
        combined[f"std_{country}_{task}"] for task in TASKS
    )

    # Standardize NRCA
    combined[f"std_{country}_NRCA"] = weighted_standardize(combined[f"{country}_NRCA"], weights)

    # Weighted NRCA for aggregation over time
    combined[f"multip_{country}_NRCA"] = combined[f"std_{country}_NRCA"] * weights

In [ ]:
# --- AGGREGATE OVER TIME AND PLOT ---

for country in COUNTRIES:
    agg = combined.groupby("TIME")[f"multip_{country}_NRCA"].sum().reset_index()

    plt.figure()
    plt.plot(agg["TIME"], agg[f"multip_{country}_NRCA"])
    plt.xticks(range(0, len(agg), 3), agg["TIME"][::3], rotation=45)
    plt.title(f"Non-Routine Cognitive Analytical Task Intensity — {country}")
    plt.xlabel("Time")
    plt.ylabel("NRCA Index")
    plt.tight_layout()
    plt.show()